In [4]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ==============================================================
# MODEL 3 — RANDOM FOREST PRICE CLASSIFIER
# CHEAP / NORMAL / EXPENSIVE
# ==============================================================

print("=" * 70)
print("MODEL 3 — RANDOM FOREST PRICE CLASSIFIER")
print("=" * 70)


# ==============================================================
# STEP 1 — LOAD DATA
# ==============================================================

df = pd.read_csv(
    "../data/raw/final_dataset.csv"
)

print("\nDataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ==============================================================
# STEP 2 — DATA TYPES
# ==============================================================

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

df["Modal_Price"] = pd.to_numeric(
    df["Modal_Price"],
    errors="coerce"
)


# ==============================================================
# STEP 3 — SORT DATA
# ==============================================================

df = df.sort_values(
    ["Commodity", "Date"]
).reset_index(drop=True)


## ==============================================================
# STEP 4 — CREATE PRICE CATEGORY
# ==============================================================

df["Price_Category"] = "Normal"

commodities = [
    "Onion",
    "Potato",
    "Rice",
    "Tomato",
    "Wheat"
]

for commodity in commodities:

    mask = df["Commodity"] == commodity

    q33 = df.loc[
        mask,
        "Modal_Price"
    ].quantile(0.33)

    q66 = df.loc[
        mask,
        "Modal_Price"
    ].quantile(0.66)

    df.loc[
        mask & (df["Modal_Price"] <= q33),
        "Price_Category"
    ] = "Cheap"

    df.loc[
        mask
        & (df["Modal_Price"] > q33)
        & (df["Modal_Price"] <= q66),
        "Price_Category"
    ] = "Normal"

    df.loc[
        mask & (df["Modal_Price"] > q66),
        "Price_Category"
    ] = "Expensive"


# ==============================================================
# STEP 5 — CHECK PRICE CATEGORY
# ==============================================================

print("\n" + "=" * 70)
print("PRICE CATEGORY DISTRIBUTION")
print("=" * 70)

print(
    df[
        ["Commodity", "Price_Category"]
    ]
    .value_counts()
    .sort_index()
)

print("\nOverall categories:")

print(
    df["Price_Category"].value_counts()
)


# ==============================================================
# CHECK COMMODITY COLUMN
# ==============================================================

print("\nCommodity column exists:")

print(
    "Commodity" in df.columns
)

print("\nColumns after category creation:")

print(df.columns.tolist())

# ==============================================================
# STEP 6 — FEATURE LIST
# ==============================================================

features = [
    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Rolling_7",
    "Rolling_30",
    "Arrival",
    "Max_Temp",
    "Min_Temp",
    "Rainfall_mm",
    "WindSpeed",
    "Month",
    "DayOfWeek",
    "Quarter",
    "Weekend",
    "Holiday"
]

target = "Price_Category"


# ==============================================================
# STEP 7 — FEATURE CHECK
# ==============================================================

print("\n" + "=" * 70)
print("FEATURE CHECK")
print("=" * 70)

for feature in features:

    print(
        f"{feature:15} ->",
        feature in df.columns
    )

print(
    f"{target:15} ->",
    target in df.columns
)


# ==============================================================
# STEP 8 — MISSING VALUE CHECK
# ==============================================================

required_columns = features + [target]

df_model = df[
    required_columns
].copy()

before = len(df_model)

df_model = df_model.dropna()

after = len(df_model)

print("\n" + "=" * 70)
print("MISSING VALUE CHECK")
print("=" * 70)

print("Rows before :", before)
print("Rows after  :", after)
print("Rows removed:", before - after)

print(
    "\nMissing values remaining:",
    df_model.isnull().sum().sum()
)


# ==============================================================
# STEP 9 — CHECK EACH COMMODITY
# ==============================================================

commodities = [
    "Onion",
    "Potato",
    "Rice",
    "Tomato",
    "Wheat"
]

print("\n" + "=" * 70)
print("COMMODITY-WISE DATA")
print("=" * 70)

for commodity in commodities:

    # Get original rows belonging to commodity
    commodity_rows = df[
        df["Commodity"] == commodity
    ]

    print("\n" + "=" * 60)
    print(commodity)
    print("=" * 60)

    print(
        "Rows:",
        len(commodity_rows)
    )

    print(
        commodity_rows[
            "Price_Category"
        ].value_counts()
    )

print("\n" + "=" * 70)
print("MODEL 3 STEP 1 COMPLETED")
print("=" * 70)

MODEL 3 — RANDOM FOREST PRICE CLASSIFIER

Dataset shape: (54340, 28)

Columns:
['State', 'Commodity_Group', 'Commodity', 'Date', 'Arrival', 'Arrival_Unit', 'Modal_Price', 'Price_Unit', 'District', 'Max_Temp', 'Min_Temp', 'Rainfall_mm', 'Rain_mm', 'WindSpeed', 'Festival', 'Holiday', 'Month', 'Year', 'Day', 'Weekend', 'DayOfWeek', 'Quarter', 'Lag_1', 'Lag_2', 'Lag_3', 'Rolling_7', 'Rolling_30', 'Price_Change']

PRICE CATEGORY DISTRIBUTION
Commodity  Price_Category
Onion      Cheap             3600
           Expensive         3700
           Normal            3590
Potato     Cheap             3590
           Expensive         3690
           Normal            3590
Rice       Cheap             3570
           Expensive         3670
           Normal            3560
Tomato     Cheap             3590
           Expensive         3690
           Normal            3590
Wheat      Cheap             3600
           Expensive         3710
           Normal            3600
Name: count, dtype: int

In [5]:
# ==============================================================
# STEP 2 — TIME-BASED TRAIN / TEST SPLIT
# ==============================================================

print("\n" + "=" * 70)
print("STEP 2 — TIME-BASED TRAIN / TEST SPLIT")
print("=" * 70)


# We need Date for chronological splitting
model_data = df[
    features
    + [target, "Commodity", "Date"]
].copy()


# Remove missing values
model_data = model_data.dropna()


# --------------------------------------------------------------
# Store data separately for each commodity
# --------------------------------------------------------------

train_data = {}
test_data = {}

for commodity in commodities:

    commodity_data = model_data[
        model_data["Commodity"] == commodity
    ].copy()

    # Sort chronologically
    commodity_data = commodity_data.sort_values(
        "Date"
    )

    # 2023-2024 = TRAIN
    train = commodity_data[
        commodity_data["Date"] < "2025-01-01"
    ].copy()

    # 2025 = TEST
    test = commodity_data[
        commodity_data["Date"] >= "2025-01-01"
    ].copy()

    train_data[commodity] = train
    test_data[commodity] = test

    print("\n" + "=" * 60)
    print(commodity)
    print("=" * 60)

    print(
        "Train rows:",
        len(train)
    )

    print(
        "Test rows :",
        len(test)
    )

    print(
        "Train dates:",
        train["Date"].min(),
        "to",
        train["Date"].max()
    )

    print(
        "Test dates :",
        test["Date"].min(),
        "to",
        test["Date"].max()
    )

    print("\nTraining classes:")

    print(
        train[target]
        .value_counts()
    )

    print("\nTesting classes:")

    print(
        test[target]
        .value_counts()
    )


print("\n" + "=" * 70)
print("STEP 2 COMPLETED")
print("=" * 70)


STEP 2 — TIME-BASED TRAIN / TEST SPLIT

Onion
Train rows: 7240
Test rows : 3650
Train dates: 2023-01-01 00:00:00 to 2024-12-31 00:00:00
Test dates : 2025-01-01 00:00:00 to 2025-12-31 00:00:00

Training classes:
Price_Category
Expensive    3030
Cheap        2160
Normal       2050
Name: count, dtype: int64

Testing classes:
Price_Category
Normal       1540
Cheap        1440
Expensive     670
Name: count, dtype: int64

Potato
Train rows: 7230
Test rows : 3640
Train dates: 2023-01-01 00:00:00 to 2024-12-31 00:00:00
Test dates : 2025-01-01 00:00:00 to 2025-12-31 00:00:00

Training classes:
Price_Category
Expensive    3010
Cheap        2870
Normal       1350
Name: count, dtype: int64

Testing classes:
Price_Category
Normal       2240
Cheap         720
Expensive     680
Name: count, dtype: int64

Rice
Train rows: 7160
Test rows : 3640
Train dates: 2023-01-01 00:00:00 to 2024-12-31 00:00:00
Test dates : 2025-01-01 00:00:00 to 2025-12-31 00:00:00

Training classes:
Price_Category
Cheap        

In [6]:
# ==============================================================
# STEP 3 — TRAIN RANDOM FOREST FOR ALL 5 COMMODITIES
# ==============================================================

print("\n" + "=" * 70)
print("STEP 3 — RANDOM FOREST TRAINING")
print("=" * 70)

models = {}
X_train_data = {}
X_test_data = {}
y_train_data = {}
y_test_data = {}

for commodity in commodities:

    print("\n" + "=" * 60)
    print(commodity)
    print("=" * 60)

    train = train_data[commodity]
    test = test_data[commodity]

    # ----------------------------------------------------------
    # Features
    # ----------------------------------------------------------

    X_train = train[features]
    X_test = test[features]

    # ----------------------------------------------------------
    # Target
    # ----------------------------------------------------------

    y_train = train[target]
    y_test = test[target]

    # ----------------------------------------------------------
    # Save datasets
    # ----------------------------------------------------------

    X_train_data[commodity] = X_train
    X_test_data[commodity] = X_test

    y_train_data[commodity] = y_train
    y_test_data[commodity] = y_test

    print("X_train:", X_train.shape)
    print("X_test :", X_test.shape)

    print("y_train:", y_train.shape)
    print("y_test :", y_test.shape)

    # ----------------------------------------------------------
    # Random Forest
    # ----------------------------------------------------------

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    # ----------------------------------------------------------
    # Train
    # ----------------------------------------------------------

    rf.fit(
        X_train,
        y_train
    )

    models[commodity] = rf

    print("Training complete.")

print("\n" + "=" * 70)
print("ALL 5 RANDOM FOREST MODELS TRAINED")
print("=" * 70)

print("\nModels:")

print(
    models.keys()
)


STEP 3 — RANDOM FOREST TRAINING

Onion
X_train: (7240, 15)
X_test : (3650, 15)
y_train: (7240,)
y_test : (3650,)
Training complete.

Potato
X_train: (7230, 15)
X_test : (3640, 15)
y_train: (7230,)
y_test : (3640,)
Training complete.

Rice
X_train: (7160, 15)
X_test : (3640, 15)
y_train: (7160,)
y_test : (3640,)
Training complete.

Tomato
X_train: (7240, 15)
X_test : (3630, 15)
y_train: (7240,)
y_test : (3630,)
Training complete.

Wheat
X_train: (7270, 15)
X_test : (3640, 15)
y_train: (7270,)
y_test : (3640,)
Training complete.

ALL 5 RANDOM FOREST MODELS TRAINED

Models:
dict_keys(['Onion', 'Potato', 'Rice', 'Tomato', 'Wheat'])


In [7]:
# ==============================================================
# STEP 4 — PREDICTIONS + MODEL PERFORMANCE
# ==============================================================

print("\n" + "=" * 70)
print("STEP 4 — RANDOM FOREST PERFORMANCE")
print("=" * 70)

results = {}

predictions = {}
probabilities = {}

for commodity in commodities:

    print("\n" + "=" * 60)
    print(commodity)
    print("=" * 60)

    # ----------------------------------------------------------
    # Get model and test data
    # ----------------------------------------------------------

    model = models[commodity]

    X_test = X_test_data[commodity]
    y_test = y_test_data[commodity]

    # ----------------------------------------------------------
    # Predictions
    # ----------------------------------------------------------

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)

    predictions[commodity] = y_pred
    probabilities[commodity] = y_prob

    # ----------------------------------------------------------
    # Metrics
    # ----------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    # ----------------------------------------------------------
    # ROC-AUC
    # ----------------------------------------------------------

    try:

        roc_auc = roc_auc_score(
            y_test,
            y_prob,
            multi_class="ovr",
            average="weighted"
        )

    except ValueError:

        roc_auc = np.nan

    # ----------------------------------------------------------
    # Save results
    # ----------------------------------------------------------

    results[commodity] = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "ROC_AUC": roc_auc
    }

    # ----------------------------------------------------------
    # Print metrics
    # ----------------------------------------------------------

    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"F1 Score : {f1:.4f}"
    )

    print(
        f"ROC-AUC  : {roc_auc:.4f}"
    )

    # ----------------------------------------------------------
    # Confusion Matrix
    # ----------------------------------------------------------

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[
            "Cheap",
            "Normal",
            "Expensive"
        ]
    )

    print("\nConfusion Matrix:")

    print(cm)

    # ----------------------------------------------------------
    # Classification Report
    # ----------------------------------------------------------

    print("\nClassification Report:")

    print(
        classification_report(
            y_test,
            y_pred,
            labels=[
                "Cheap",
                "Normal",
                "Expensive"
            ],
            zero_division=0
        )
    )


# ==============================================================
# FINAL PERFORMANCE TABLE
# ==============================================================

performance_df = pd.DataFrame(
    results
).T.reset_index()

performance_df = performance_df.rename(
    columns={
        "index": "Commodity"
    }
)


print("\n" + "=" * 70)
print("RANDOM FOREST CLASSIFIER — FINAL PERFORMANCE")
print("=" * 70)

print(
    performance_df.to_string(
        index=False
    )
)


STEP 4 — RANDOM FOREST PERFORMANCE

Onion
Accuracy : 0.9852
Precision: 0.9852
Recall   : 0.9852
F1 Score : 0.9852
ROC-AUC  : 0.9959

Confusion Matrix:
[[1421   18    1]
 [  19 1517    4]
 [   1   11  658]]

Classification Report:
              precision    recall  f1-score   support

       Cheap       0.99      0.99      0.99      1440
      Normal       0.98      0.99      0.98      1540
   Expensive       0.99      0.98      0.99       670

    accuracy                           0.99      3650
   macro avg       0.99      0.98      0.99      3650
weighted avg       0.99      0.99      0.99      3650


Potato
Accuracy : 0.9563
Precision: 0.9568
Recall   : 0.9563
F1 Score : 0.9565
ROC-AUC  : 0.9857

Confusion Matrix:
[[ 676   43    1]
 [  70 2150   20]
 [   0   25  655]]

Classification Report:
              precision    recall  f1-score   support

       Cheap       0.91      0.94      0.92       720
      Normal       0.97      0.96      0.96      2240
   Expensive       0.97      